# TẬP DƯỢT PHẢN BIỆN & RÀ SOÁT LỖ HỔNG (GAP ANALYSIS) TRƯỚC HỘI ĐỒNG

**Mục đích:** Đóng vai **một giảng viên phản biện cực kỳ khắt khe** để tìm ra **15 lỗ hổng / điểm yếu / câu hỏi xoáy sâu thực sự tồn tại** trong đồ án **Vi-ESG-Bert** (bám sát số liệu, code và bảng biểu thật của chính bạn), kèm:
1. **Bẫy cần tránh** (những kiểu trả lời dễ bị mất điểm hoặc bị giảng viên bắt bẻ tiếp).
2. **Khung trả lời phản biện 3 bước chuẩn học thuật:** *Thừa nhận khách quan phần đúng → Đưa bằng chứng định lượng đã có → Chốt giải pháp / hướng khắc phục*.
3. **Bộ mô phỏng Tập dượt Hỏi–Đáp tương tác (`ipywidgets`) ở cuối notebook:** Rút ngẫu nhiên câu hỏi, tự tập trả lời thành tiếng trước khi mở đáp án, và tự chấm điểm tiến độ ôn luyện `X / 15` câu!

> **Nguyên tắc vàng khi bảo vệ đồ án khoa học:** Không bao giờ cãi cố hoặc che giấu giới hạn nghiên cứu. Giảng viên đánh giá **rất cao** sinh viên nhìn rõ giới hạn của bài mình, giải thích rành mạch bằng số liệu thực nghiệm và đề xuất được hướng xử lý hợp lý.

---
## NHÓM 1 — Lỗ hổng Dữ liệu & Kiểm soát Rò rỉ (Data & Leakage Gaps)

### Gap 1 `[Mức độ: Khó]` — Kiểm toán trùng lặp mới là khớp chuỗi chính xác, lỡ có câu gần giống (*Near-Duplicate*) thì sao?
- **Giảng viên hỏi xoáy:** *"Bạn khẳng định `0` mẫu rò rỉ giữa `train`, `test` và `unlabeled`, nhưng bạn chỉ so khớp chuỗi sau khi bỏ dấu câu và viết thường (`exact normalized match`). Lỡ có hai câu chỉ khác 1–2 từ hoặc trích từ cùng một báo cáo thường niên của cùng một doanh nghiệp rơi vào cả `train` và `test` thì sao?"*
- **Bẫy cần tránh:** Khẳng định bừa rằng chắc chắn không có câu nào giống nhau.
- **Cách trả lời chuẩn mực:**
  1. **Thừa nhận thẳng:** Đúng là bước kiểm toán B2 chỉ kiểm soát trùng lặp chính xác sau chuẩn hóa Unicode NFC và loại dấu câu; đồng thời bộ dữ liệu gốc không cung cấp siêu dữ liệu *Mã doanh nghiệp / Năm báo cáo* nên không thể chia theo nhóm tài liệu (*Group Split*).
  2. **Bằng chứng bổ trợ:** Tuy nhiên, kết quả EDA (Hình `B2_02`) cho thấy phân phối độ dài và tỷ lệ lớp của `test.csv` khác rõ rệt so với `train_pool` (`Mean = 44.3` vs `37.9` từ; `Environment 27.3%` vs `16.0%`), chứng tỏ `test.csv` đến từ một nguồn/lát cắt phân phối khác chứ không đơn thuần là cắt ngẫu nhiên từ cùng văn bản với `train`.
  3. **Hướng bổ sung:** Nếu mở rộng thu thập dữ liệu mới, sẽ áp dụng thêm lọc tương đồng ngữ nghĩa (*MinHash / Cosine Embedding Similarity ≥ 0.95*) và chia theo *Company/Document ID*.

---
### Gap 2 `[Mức độ: Khó]` — Tập `test.csv` có dấu hiệu nhiễu nhãn (`row 2474, 413, 185, 1022`), sao không sửa nhãn test rồi chấm lại?
- **Giảng viên hỏi xoáy:** *"Trong phần Phân tích lỗi bạn nói một số câu ở `test.csv` bị gán nhãn gốc chưa hợp lý. Vậy có phải bạn đang đổ lỗi cho dữ liệu không? Sao không sửa lại nhãn rồi đánh giá lại?"*
- **Bẫy cần tránh:** Nói rằng bộ dữ liệu sai nhiều nên mô hình thực ra đạt 95%+.
- **Cách trả lời chuẩn mực:**
  1. **Nguyên tắc chuẩn mực:** Tuyệt đối **không được tự ý sửa nhãn tập kiểm thử (`test.csv`)**, vì làm vậy sẽ phá vỡ tính khách quan so sánh chuẩn (*benchmark integrity*) và dễ bị thiên vị xác nhận (*cherry-picking*).
  2. **Bằng chứng cụ thể:** Nhóm giữ nguyên 100% nhãn gốc của `3.000` câu `test.csv` để báo cáo điểm chính thức (`90.77%` Acc / `90.09%` Macro-F1). Việc trích dẫn trực tiếp nguyên văn dòng `2474` (*"trưng bày công nghệ xanh, năng lượng tái tạo..."* gán `Irrelevant`) hay dòng `413` (*"94% bao bì tái chế, phân hủy"* gán `Irrelevant`) chỉ nhằm diễn giải khách quan nguyên nhân của các ca sai độ tự tin `> 99.8%`, không dùng để thổi phồng chỉ số.

---
### Gap 3 `[Mức độ: Trung bình]` — Dữ liệu chứa nhiều câu dịch từ báo cáo nước ngoài, đem áp dụng lên báo cáo doanh nghiệp Việt Nam có bị rớt điểm không?
- **Giảng viên hỏi xoáy:** *"Đọc mẫu lỗi thấy có câu nhắc tới 'Nedbank', 'người Mỹ gốc Phi', văn phong dịch máy. Vậy mô hình này đem đọc báo cáo phát triển bền vững của doanh nghiệp Việt Nam thuần túy (Vinamilk, FPT, Hòa Phát) có chạy ổn không?"*
- **Cách trả lời chuẩn mực:**
  1. **Thừa nhận đặc trưng dữ liệu:** Bộ dữ liệu công khai của bài toán quả thực bao gồm nhiều câu dịch từ báo cáo ESG quốc tế (đây cũng là một phần lý do khiến `DistilBERT-m` và `mBERT-cased` đa ngôn ngữ hoạt động rất tốt).
  2. **Chứng minh & Hướng mở rộng:** Trong notebook `DEMO_TRUC_TIEP_CHO_GIANG_VIEN.ipynb`, nhóm đã thử nghiệm trực tiếp trên các câu viết theo văn phong báo cáo doanh nghiệp Việt Nam thực tế và mô hình nhận diện chính xác với đồng thuận `7/7`. Tuy vậy, để triển khai sản xuất tại thị trường Việt Nam, cần xây dựng thêm một tập kiểm thử độc lập trích từ báo cáo phát triển bền vững của các doanh nghiệp niêm yết VN30 để đánh giá *out-of-domain generalization*.

---
## NHÓM 2 — Lỗ hổng Thiết kế Thực nghiệm & Đối chứng (Ablation & Methodology Gaps)

### Gap 4 `[Mức độ: CỰC GẮT — BẮT BUỘC NẮM VỮNG!]` — Chưa có `Baseline Ensemble (7 mô hình)` và chưa có `DistilBERT-m Baseline`!
- **Giảng viên hỏi xoáy:** *"Bạn so sánh `Equal Soft Ensemble (7M)` (`90.09%`) với `Best Baseline PhoBERT` (`86.46%`) rồi nói tăng `+3.63%`. Nhưng đó là so **7 mô hình + nhãn giả** với **1 mô hình đơn lẻ không nhãn giả**! Nhỡ bạn chạy Ensemble 7 mô hình trên 100% `train_pool` gốc (không cần Pseudo-Labeling) nó cũng đạt `89.5%` thì sao? Hơn nữa `DistilBERT-m` (mô hình mạnh nhất Run 3) bạn lại không chạy bản Baseline!"*
- **Bẫy cần tránh:** Cố biện minh rằng Baseline 1 mô hình là đủ để kết luận cho cả Ensemble 7 mô hình.
- **Cách trả lời bản lĩnh (Ăn điểm tuyệt đối):**
  1. **Thừa nhận thẳng đây là một điểm thiếu sót hợp lý trong thiết kế Ablation** (do giới hạn tài nguyên/thời gian trên 1 GPU cá nhân, nhóm chạy đối chứng Baseline cho 3 kiến trúc tiếng Việt chủ lực thay vì đủ cả 7 mô hình + Baseline Ensemble).
  2. **Bóc tách công bằng bằng chính bằng chứng 1-chọi-1 đã có:** Khi so sánh **cùng 1 kiến trúc** (cùng không Ensemble), cả **3/3 mô hình có đối chứng ở Run 3 đều đánh bại bản Baseline 100% `train_pool` gốc**: `PhoBERT` `+1.67%`, `Vi-ELECTRA` `+1.50%`, `ViSoBERT` `+4.56%` — trung bình tăng **`+2.58%` Macro-F1** cho mỗi mô hình đơn lẻ! Đồng thời Baseline có `ECE = 0.111–0.136` (quá tự tin nặng do lệch lớp `IR = 2.29`).
  3. **Kết luận chặt chẽ:** Mức tăng `+3.63%` của `Equal Soft (7M)` là tổng hợp đóng góp của **hai thành phần rõ rệt**: phần tăng từ **Pseudo-Labeling cân bằng lớp** (chứng minh qua so sánh 1-chọi-1 ở từng mô hình) cộng thêm phần tăng từ **Ensemble 7 kiến trúc** (`+1.06%`, `p = 0.0161` so với Best Single Run 3). Nếu có thêm thời gian tính toán, chạy đủ `Baseline 7M Ensemble` sẽ giúp tách 2 thành phần này hoàn toàn trực giao.

---
### Gap 5 `[Mức độ: Trung bình]` — Tại sao Vòng 1 chỉ train trên tập con `10.000` mẫu thay vì `24.628` mẫu gốc?
- **Giảng viên hỏi xoáy:** *"Bạn có sẵn `24.628` mẫu sạch mà Vòng 1 lại cắt xuống còn `10.000` mẫu cho mỗi mô hình, có phải cố tình dìm Vòng 1 để Vòng 2–3 nhìn tăng mạnh hơn không?"*
- **Cách trả lời chuẩn mực:**
  1. **Không hề cố tình dìm mốc so sánh**, bởi chính vì lo ngại câu hỏi này nên nhóm đã bổ sung riêng **Ablation Baseline học 100% `24.628` mẫu** để so sánh công bằng!
  2. **Hai lý do kỹ thuật của việc chia 7 tập con 10k ở Vòng 1:** (i) tạo sự khác biệt dữ liệu (*subsampling/bagging*) giúp 7 mô hình đa dạng lỗi ngay từ vòng đầu; (ii) thực nghiệm chứng minh một điều bất ngờ: `Vi-ELECTRA` học 10k mẫu ở Run 1 (`87.29%` Macro-F1, `ECE = 0.0576`) còn tổng quát hóa tốt hơn bản Baseline học 24.6k mẫu lệch lớp (`85.81%`, `ECE = 0.1157`). Đến Vòng 2 và Vòng 3, **100% `24.628` mẫu gốc đều được giữ nguyên toàn bộ** khi gộp nhãn giả.

---
### Gap 6 `[Mức độ: Khó]` — Các ngưỡng lọc (`0.94 / 0.50`, `0.91 / 0.45`) và mục tiêu `9.500 / 10.500` chọn theo cảm tính (heuristic)? Tại sao không chạy tiếp Vòng 4?
- **Giảng viên hỏi xoáy:** *"Tại sao là `0.94` rồi `0.91` chứ không phải `0.95 / 0.85`? Sao bạn không thử chạy tiếp Vòng 3 Pseudo-Labeling (Run 4) xem điểm có tăng nữa không?"*
- **Cách trả lời chuẩn mực:**
  1. Các ngưỡng được đặt theo nguyên tắc *curriculum thresholding* (Vòng 1 rất chặt `0.94` để lấy lõi sạch nhất; Vòng 2 hạ nhẹ `0.91` khi cả 7/7 mô hình đã hội tụ đồng đều và có `97.52%` mẫu đạt đồng thuận 7/7) kết hợp quan sát trực tiếp phân phối `1 - ECDF` (`Hình B4_01` & `B6_03`).
  2. **Lý do dừng hợp lý ở 2 vòng Pseudo-Labeling (Run 3):** Sau Vòng 2, `60.4%` (`32.903`) tập không nhãn đã được khai thác và `IR` đã giảm về **`1.35`** (khớp sát `1.42` của Test). Phần `39.6%` (`21.584` câu) còn lại có độ bất định cao; nếu tiếp tục hạ ngưỡng xuống `< 0.85` ở vòng sau, nguy cơ đưa nhiễu nhãn (*confirmation bias*) sẽ tăng nhanh hơn lợi ích thu được. Tuy vậy, thực hiện phân tích độ nhạy siêu tham số (*hyperparameter sensitivity*) cho $\tau$ là một hướng mở rộng rất tốt.

---
### Gap 7 `[Mức độ: Khó]` — Toàn bộ thực nghiệm chỉ chạy với 1 seed duy nhất (`seed = 42`), có chắc kết quả không bị phụ thuộc seed?
- **Giảng viên hỏi xoáy:** *"Các bài báo NLP chuẩn thường chạy 3–5 seed rồi báo `mean ± std`. Bạn chỉ chạy 1 seed (`42`), lỡ `DistilBERT-m` cao hơn `PhoBERT` hay Run 3 tăng điểm chỉ do may mắn seed thì sao?"*
- **Cách trả lời chuẩn mực:**
  1. Thừa nhận việc chỉ chạy 1 seed là giới hạn về ngân sách tính toán trên 1 GPU cá nhân (`24` lượt huấn luyện đầy đủ cho Baseline + 3 vòng). Nhóm cố định `seed = 42` toàn hệ thống để bảo đảm tính tái lập hoàn toàn.
  2. **Bằng chứng giảm thiểu rủi ro may mắn:** Xu hướng tăng từ Run 2 → Run 3 và vượt Baseline xảy ra **đồng loạt trên cả 7/7 mô hình độc lập** (không phải 1 mô hình cá biệt), và **kiểm định McNemar** trên toàn bộ `3.000` mẫu kiểm thử xác nhận `Equal Soft (7M)` vượt `Best Single` (`p = 0.0161`) và `Best Baseline` (`p < 10^{-6}`) có ý nghĩa thống kê.

---
## NHÓM 3 — Lỗ hổng Thuật toán, Hiệu chuẩn & Bất thường Số liệu (Algorithm & Result Gaps)

### Gap 8 `[Mức độ: Khó]` — Vì sao `Weighted Soft` lại thua `Equal Soft`, và `Top-3 Weighted` lại bỏ sót mô hình mạnh nhất `DistilBERT-m`?
- **Giảng viên hỏi xoáy:** *"Bạn thiết kế trọng số theo `Val Macro-F1`, sao cuối cùng chia đều `1/7` (`90.09%`) lại thắng trọng số (`90.04%`), còn `Top-3` lại không chọn `DistilBERT-m`?"*
- **Cách trả lời chuẩn mực:**
  1. Đây chính là **bằng chứng trực tiếp của Zero Data Leakage**: nhóm tuyệt đối không nhìn `test.csv` để chọn thành viên `Top-3` hay tính trọng số, mà chỉ dựa duy nhất trên `val.csv` — nơi `PhoBERT (91.10%)`, `Vi-ELECTRA (90.55%)` và `mBERT (89.82%)` đứng đầu.
  2. Vì `val.csv` có phân phối giống `train_pool` (`IR = 2.28`), nên trọng số theo `Val` hơi ưu ái các mô hình khớp sát miền train hơn một chút (`Val − Test` của `PhoBERT` là `+2.97%`, còn `DistilBERT-m` chỉ `+0.34%`). Chia đều `1/7` đóng vai trò **điều chuẩn không thiên vị**, giúp tận dụng tốt hơn độ hiệu chuẩn cao của `DistilBERT-m`, `RoBERTa-base` và `ViDeBERTa-base` trên miền Test.

---
### Gap 9 `[Mức độ: Khó — RẤT DỄ BỊ BẮT LỖI NHẤT QUÁN]` — Công thức Quality Gate loại mô hình `< 0.50`, nhưng ở B4 `ViDeBERTa-base` vẫn có trọng số `0.0253`!
- **Giảng viên hỏi xoáy:** *"Trong bài báo bạn viết công thức Quality Gate đặt trọng số = 0 nếu `Val F1 < 0.50`. Nhưng trong notebook B4 mình thấy bảng log `ViDeBERTa-base` vẫn có trọng số `0.0253` (không phải 0). Có mâu thuẫn giữa code thực tế và lý thuyết không?"*
- **Cách trả lời trung thực & thuyết phục:**
  1. **Trả lời trung thực đúng lịch sử triển khai:** Ở lần chạy B4 ban đầu, trọng số được tính thuần tỷ lệ `Val F1` nên `ViDeBERTa-base` (`13.41%`) tự bị hạ xuống chỉ còn `2.53%` (mức ảnh hưởng rất nhỏ vì `23.378` nhãn giả lọc ra vẫn đạt Avg Conf `0.9682` và Avg Margin `0.9533`).
  2. **Cải tiến kỹ thuật rút ra từ chính sự cố đó:** Ngay sau khi phát hiện nguyên nhân collapse `float16` ở B3–B4, nhóm đã lập trình bổ sung cơ chế **Quality Gate (`min_model_val_f1 = 0.50`)** vào `src/ensemble.py` cho vòng 2 trở đi và đưa vào khuôn khổ phương pháp tổng quát để hệ thống miễn nhiễm hoàn toàn với sự cố tương tự khi tái chạy.

---
### Gap 10 `[Mức độ: Khó]` — Tại sao không dùng `Class-Weighted Loss` / `Focal Loss` / `Oversampling` đơn giản hơn nhiều?
- **Giảng viên hỏi xoáy:** *"Mất cân bằng lớp `IR = 2.29` khá nhẹ, sao không dùng `Class-Weighted Cross-Entropy` hay `Focal Loss` cho nhanh mà phải chạy 3 vòng Pseudo-Labeling phức tạp?"*
- **Cách trả lời sắc bén:**
  1. `Class-Weighted Loss`, `Focal Loss` hay `Oversampling` chỉ **tái sử dụng lặp đi lặp lại đúng `3.945` câu `Environment` cũ** — không cung cấp thêm bất kỳ tri thức từ vựng hay cấu trúc câu mới nào, đồng thời hay làm méo xác suất dự đoán (tăng `ECE`).
  2. Bộ dữ liệu của chúng ta không chỉ bị lệch lớp (`IR = 2.29`), mà còn bị **Covariate Length Shift** (câu Test dài hơn câu Train: `44.3` vs `37.9` từ) và có sẵn **`54.487` câu không nhãn dài hơn (`48.3` từ)**. Selective Pseudo-Labeling giải quyết **đồng thời 3 việc**: mở rộng kho mẫu mới, thích nghi câu dài, và cân bằng lại phân phối tiên nghiệm.

---
### Gap 11 `[Mức độ: CỰC SẮC]` — Muốn giảm `ECE` chỉ cần `Temperature Scaling` trên 1 mô hình, cần gì chạy 7 mô hình Ensemble?
- **Giảng viên hỏi xoáy:** *"Bạn trích dẫn *Guo et al. (2017)* về Calibration, chính bài báo đó chứng minh chỉ cần học 1 tham số nhiệt độ $T$ (*Temperature Scaling*) trên `val.csv` là giảm `ECE` của 1 mô hình xuống `0.02` ngay, đâu cần Ensemble 7 mô hình tốn kém?"*
- **Cách trả lời chuẩn xác 100%:**
  1. **Đồng ý ngay về mặt toán học:** Đúng là *Temperature Scaling* có thể giảm `ECE` rất hiệu quả cho `DistilBERT-m` đơn lẻ (là bước nên thêm vào nếu triển khai bản 1 mô hình `DistilBERT-m`).
  2. **Điểm khác biệt cốt lõi:** *Temperature Scaling* là phép chia logit cho hằng số dương $T > 0$ nên **không bao giờ thay đổi `argmax`**, tức là **không tăng được 0.01% Accuracy hay Macro-F1 nào** (`DistilBERT-m` vẫn giữ nguyên `89.03%`). Trong khi đó, `Equal Soft (7M)` đem lại **hiệu ứng kép**: vừa tăng thực chất Macro-F1 lên **`90.09%`** (`+1.06%`, `p = 0.0161`), vừa tự động giảm `ECE` xuống **`0.0226`** mà không cần tối ưu tham số hậu kỳ.

---
## NHÓM 4 — Lỗ hổng Mô hình hóa Bài toán & So sánh Hiện đại (Formulation & LLM Gaps)

### Gap 12 `[Mức độ: Rất hay bị hỏi]` — Thời đại LLM (GPT-4o / Gemini / Qwen / Llama), tại sao vẫn dùng các mô hình BERT-family?
- **Giảng viên hỏi xoáy:** *"Hiện nay chỉ cần viết 1 prompt gọi GPT-4o hay Gemini hoặc fine-tune LoRA Qwen-7B là phân loại ESG được ngay, tại sao bạn vẫn dùng 7 mô hình Encoder nhỏ thế hệ BERT?"*
- **Cách trả lời thuyết phục 4 ý:**
  1. **Tốc độ & Chi phí:** `DistilBERT-m` xử lý **`3.085` câu/giây (`0.324 ms/câu`)**, cả 7M Ensemble chạy **`234` câu/giây** cục bộ với chi phí API = `0` đồng; gọi LLM cho `54.487` câu không nhãn + báo cáo hàng chục nghìn câu sẽ chậm hơn hàng trăm lần và tốn chi phí lớn.
  2. **Bảo mật dữ liệu nội bộ (On-Premise):** Báo cáo tài chính/quản trị chưa công bố của doanh nghiệp không thể đẩy ra máy chủ API bên ngoài; kiến trúc Encoder gọn nhẹ triển khai hoàn toàn nội bộ trên 1 GPU tầm trung.
  3. **Hiệu chuẩn xác suất định lượng (`ECE = 0.0226`):** Encoder xuất trực tiếp phân phối xác suất softmax đã hiệu chuẩn cho cơ chế Human-in-the-Loop, điều mà LLM sinh văn bản khó đảm bảo ổn định.
  4. **Không đối nghịch mà bổ trợ:** Trong hướng phát triển, có thể kết hợp LLM làm *Teacher* hoặc phân loại ngữ cảnh đoạn dài để chưng cất về Encoder gọn nhẹ.

---
### Gap 13 `[Mức độ: Trung bình]` — Precision của `Governance` chỉ đạt `82.54%` (`85` câu `Irrelevant` bị nhận nhầm sang `Governance`), xử lý sao khi triển khai?
- **Giảng viên hỏi xoáy:** *"Nhìn Confusion Matrix thấy `Governance` có Recall `97.1%` nhưng Precision chỉ `82.5%` vì hay nhận nhầm báo cáo tài chính thuần túy (`Irrelevant`). Khi triển khai thật khách hàng bị spam cảnh báo giả thì bạn giải quyết thế nào?"*
- **Cách trả lời thực chiến:**
  1. **Giải pháp ngắn hạn (không cần train lại):** Áp dụng **tối ưu ngưỡng quyết định riêng theo từng lớp (*Class-wise Threshold Tuning*)** trên `val.csv` — yêu cầu `P(Governance)` vượt ngưỡng cao hơn mới gán `Governance`, đánh đổi vài % `Recall` đang rất dư (`97.1%`) để kéo `Precision` lên cân bằng.
  2. **Giải pháp căn cơ dài hạn:** Nâng cấp đầu vào từ *mức câu độc lập* lên *ngữ cảnh đoạn văn + tiêu đề mục* (để phân biệt chương "Thuyết minh báo cáo tài chính" với chương "Quản trị công ty") và chuyển sang gán nhãn **Đa nhãn (Multi-Label)** cho biên `Social ↔ Governance`.

---
## NHÓM 5 — Lỗ hổng Kỹ thuật & Triển khai Thực tế (Engineering & Deployment Gaps)

### Gap 14 `[Mức độ: Trung bình]` — Cắt câu tại `max_length = 256` token có làm mất nội dung câu dài (`454` từ ở Test, `796` từ ở Unlabeled) không?
- **Giảng viên hỏi xoáy:** *"Bạn cắt cứng ở 256 token, vậy mấy câu dài 400–700 từ bị chặt mất nửa sau thì mô hình đoán sai sao?"*
- **Cách trả lời bằng số liệu thực chứng:**
  1. **Tỷ lệ bị cắt cực nhỏ:** EDA Bước B2 chứng minh `99.90%` câu trên `test` (`P99 = 133` từ ≈ `190` token) và `99.60%` câu trên `unlabeled` nằm gọn trong `256` token.
  2. **Ràng buộc kiến trúc & tính công bằng:** `PhoBERT-base` có giới hạn vị trí tối đa cố định là **`256` token** (`max_position_embeddings = 258`). Đặt chung `max_length = 256` vừa đảm bảo công bằng đầu vào cho cả 7 mô hình, vừa tránh tăng chi phí chú ý $O(L^2)$ lên gấp 4 lần chỉ vì `0.1%` ngoại lai.

---
### Gap 15 `[Mức độ: Khó]` — Ensemble 7 mô hình (`988.4M` tham số) nặng gấp 7 lần — triển khai thật thì chọn bản nào?
- **Giảng viên hỏi xoáy:** *"Chẳng lẽ công ty nào cũng phải tải cả 7 mô hình nặng gần 1 tỷ tham số chỉ để phân loại 4 nhãn ESG? Như vậy quá cồng kềnh để bảo trì!"*
- **Cách trả lời gọn gàng bằng Đường biên Pareto (`Hình B8_05`):**
  1. **Hệ thống không ép dùng duy nhất 7M Ensemble**, mà cung cấp **3 điểm tối ưu trên đường biên Pareto**:
     - **API thời gian thực / chi phí thấp:** Dùng **`DistilBERT-m (Run 3)`** (`89.03%` Macro-F1, `0.324 ms/câu`, chỉ `135M` tham số / `~0.6–2.96 GB` VRAM) — đã thừa hưởng trọn vẹn tri thức của 2 vòng Ensemble Pseudo-Labeling nên mạnh hơn hẳn mọi Baseline mà nhanh gấp **13.2 lần** 7M Ensemble.
     - **Kiểm toán ESG theo lô hàng quý:** Dùng **`Equal Soft (7M)`** (`90.09%`, `ECE = 0.0226`) vì `10.000` câu báo cáo chỉ mất ~43 giây trên 1 GPU tầm trung.
  2. **Bước sản xuất hóa tối ưu tiếp theo:** Dùng `Equal Soft (7M)` làm *Teacher* để **Chưng cất tri thức (Knowledge Distillation)** ép toàn bộ sức mạnh `~90%` và độ hiệu chuẩn `ECE ≈ 0.02` vào đúng **1 mô hình Student `DistilBERT-m`** duy nhất!

---
## PHẦN THỰC HÀNH — Bộ Mô phỏng Tập dượt Phản biện Tương tác (`ipywidgets`)

Chạy cell dưới đây để luyện phản xạ trả lời trước buổi bảo vệ:
1. Chọn **Nhóm câu hỏi** (hoặc *Tất cả 15 Gap*) → bấm **"Câu hỏi ngẫu nhiên"** (hoặc **"Câu tiếp theo"**).
2. Tự trả lời thành tiếng trong **30–60 giây** (như đang đứng trước hội đồng).
3. Bấm **"Xem gợi ý trả lời & Bẫy cần tránh"** để tự đối chiếu → bấm **"Đã nắm vững"** hoặc **"Cần luyện lại"** để theo dõi điểm tập dượt!

In [ ]:
# ============================================================================
# BỘ MÔ PHỎNG TẬP DƯỢT PHẢN BIỆN HỘI ĐỒNG (15 GAP THỰC CHIẾN)
# ============================================================================
import random
import ipywidgets as widgets
from IPython.display import display

GAPS_BANK = [
    {
        "id": 1, "group": "1. Dữ liệu & Rò rỉ", "level": "Khó",
        "question": "Bạn kiểm toán 0 mẫu rò rỉ bằng so khớp chuỗi chính xác (exact normalized match). Lỡ trong train và test có các câu paraphrase gần giống hoặc trích từ cùng 1 báo cáo của cùng 1 doanh nghiệp thì sao?",
        "trap": "Khẳng định chắc nịch không có câu nào giống nhau mà không có bằng chứng.",
        "answer": "(1) Thừa nhận thẳng: B2 kiểm soát trùng khớp chính xác sau Unicode NFC + bỏ dấu câu; dữ liệu gốc không có cột Company/Document ID nên chưa thể chia Group Split. (2) Bằng chứng: EDA (B2_02) cho thấy test.csv có phân phối độ dài và tỷ lệ lớp lệch rõ so với train (Mean 44.3 vs 37.9 từ; Env 27.3% vs 16.0%), chứng tỏ test là lát cắt phân phối khác. (3) Hướng mở rộng: bổ sung lọc Near-Duplicate bằng MinHash / Cosine Embedding >= 0.95 khi thu thập dữ liệu mới.",
        "evidence": "Hình B2_01, Hình B2_02, Bảng kiểm toán B2_05",
    },
    {
        "id": 2, "group": "1. Dữ liệu & Rò rỉ", "level": "Khó",
        "question": "Bạn nói test.csv có nhiều câu bị nhiễu nhãn gốc (row 2474, 413, 185...). Sao bạn không sửa lại nhãn test rồi chấm lại? Có phải bạn đang đổ lỗi cho dữ liệu?",
        "trap": "Nói rằng dữ liệu sai nhiều nên thực tế mô hình đạt trên 95%.",
        "answer": "(1) Nguyên tắc khoa học: TUYỆT ĐỐI không tự sửa nhãn test.csv vì sẽ phá vỡ tính khách quan so sánh chuẩn (benchmark integrity) và gây cherry-picking. (2) Toàn bộ kết quả 90.77% Acc / 90.09% Macro-F1 đều chấm nguyên trạng trên 3.000 câu gốc. (3) Việc trích dẫn nguyên văn row 2474 ('công nghệ xanh, năng lượng tái tạo' gán Irrelevant) và row 413 ('94% bao bì tái chế' gán Irrelevant) chỉ nhằm phân tích minh bạch các ca sai tự tin >99.8%.",
        "evidence": "File B8_top15_errors.csv (dòng 2474, 413, 185, 1022)",
    },
    {
        "id": 3, "group": "1. Dữ liệu & Rò rỉ", "level": "Trung bình",
        "question": "Dữ liệu gốc có nhiều câu văn phong dịch từ báo cáo nước ngoài (Nedbank, Mỹ gốc Phi). Đem mô hình chạy trên báo cáo phát triển bền vững của doanh nghiệp Việt Nam thuần túy có bị rớt điểm không?",
        "trap": "Phủ nhận việc dữ liệu gốc có câu dịch máy.",
        "answer": "(1) Thừa nhận đặc điểm miền dữ liệu công khai (chứa câu dịch từ báo cáo quốc tế — đây cũng là lý do DistilBERT-m và mBERT đa ngôn ngữ tổng quát hóa tốt). (2) Kiểm chứng thực tế trên Notebook Demo với các câu thuần văn phong doanh nghiệp VN vẫn đạt đồng thuận 7/7. (3) Hướng thực tế: cần xây dựng thêm tập kiểm thử độc lập từ báo cáo phát triển bền vững VN30 để đo out-of-domain generalization.",
        "evidence": "Notebook DEMO_TRUC_TIEP_CHO_GIANG_VIEN.ipynb + B8_top15_errors.csv",
    },
    {
        "id": 4, "group": "2. Đối chứng & Phương pháp", "level": "CỰC GẮT",
        "question": "Bạn so Equal Soft (7M) (90.09%) với Best Baseline PhoBERT (86.46%) rồi nói tăng +3.63%. Đó là so 7 mô hình + nhãn giả với 1 mô hình đơn lẻ! Sao bạn không chạy Baseline Ensemble 7 mô hình và DistilBERT-m Baseline?",
        "trap": "Cãi cố rằng Baseline 1 mô hình là đủ chứng minh cho toàn bộ mức tăng +3.63%.",
        "answer": "(1) Thừa nhận thẳng đây là giới hạn thiết kế có thể mở rộng thêm do ngân sách thời gian trên 1 GPU cá nhân. (2) Phản biện bằng so sánh 1-chọi-1 công bằng tuyệt đối đã có: cả 3/3 mô hình cùng kiến trúc ở Run 3 đều vượt Baseline 100% train_pool (PhoBERT +1.67%, Vi-ELECTRA +1.50%, ViSoBERT +4.56%, trung bình +2.58% mỗi mô hình đơn lẻ), đồng thời Baseline bị quá tự tin nặng (ECE 0.11–0.136). (3) Kết luận rõ: mức tăng +3.63% là tổng hợp từ cả hai đóng góp — Pseudo-Labeling cân bằng lớp (chứng minh 1-chọi-1) VÀ Ensemble 7 kiến trúc (+1.06%, p=0.0161 so với Best Single Run 3).",
        "evidence": "Bảng tiến trình 3 vòng + Hình B8_01 + Hình B3_01b (Val Loss Baseline tăng)",
    },
    {
        "id": 5, "group": "2. Đối chứng & Phương pháp", "level": "Trung bình",
        "question": "Có sẵn 24.628 mẫu gốc mà Vòng 1 lại chia 7 tập con 10.000 mẫu, có lãng phí dữ liệu hay cố tình làm Vòng 1 yếu đi không?",
        "trap": "Nói rằng do máy yếu nên cắt bớt còn 10k.",
        "answer": "(1) Không hề dìm mốc so sánh vì nhóm đã chạy riêng Ablation Baseline trên 100% 24.628 mẫu để đối chứng. (2) Chia 7 tập con khác nhau tạo độ đa dạng lỗi (subsampling/bagging) cho Ensemble Vòng 1. (3) Thậm chí thực nghiệm còn phát hiện Vi-ELECTRA học 10k mẫu (87.29%, ECE=0.0576) tốt hơn chính nó học 24.6k mẫu lệch lớp (85.81%, ECE=0.1157). Từ Vòng 2 trở đi, 100% 24.628 mẫu gốc đều được giữ trọn vẹn.",
        "evidence": "Bảng B3 (Vi-ELECTRA Baseline vs Run 1) + Hình B6_02",
    },
    {
        "id": 6, "group": "2. Đối chứng & Phương pháp", "level": "Khó",
        "question": "Ngưỡng lọc 0.94/0.50 (Vòng 1), 0.91/0.45 (Vòng 2) và mục tiêu 9.500/10.500 chọn theo cảm tính à? Tại sao không chạy tiếp Vòng 4?",
        "trap": "Trả lời rằng thử đại thấy đẹp nên chọn.",
        "answer": "(1) Đặt ngưỡng theo nguyên lý curriculum thresholding (chặt trước để lấy lõi tinh khiết, nới nhẹ sau khi 7/7 mô hình đã hội tụ và đạt 97.52% đồng thuận 7/7) kết hợp quan sát phân phối 1-ECDF. (2) Lý do dừng ở Vòng 2 (Run 3): đã khai thác 60.4% tập không nhãn và kéo IR về 1.35 (khớp sát 1.42 của Test). 39.6% câu còn lại (21.584 câu) có độ bất định cao — hạ ngưỡng tiếp sẽ làm tăng nguy cơ nhiễu xác nhận (confirmation bias).",
        "evidence": "Hình B6_01 (Data Funnel), Hình B6_02 (IR 2.29->1.35), Hình B6_03 (1-ECDF)",
    },
    {
        "id": 7, "group": "2. Đối chứng & Phương pháp", "level": "Khó",
        "question": "Toàn bộ thí nghiệm chỉ chạy với 1 seed duy nhất (seed=42), không có mean ± std qua nhiều seed. Làm sao chắc kết quả không do hên seed?",
        "trap": "Nói rằng Transformer rất ổn định nên 1 seed là đủ chắc chắn 100%.",
        "answer": "(1) Thừa nhận thẳng giới hạn tài nguyên tính toán khi huấn luyện 24 lượt trên 1 GPU cá nhân (nhân 3–5 seeds sẽ tăng gấp 3–5 lần thời gian); cố định seed=42 để bảo đảm tái lập 100%. (2) Bằng chứng chống may mắn: mức tăng ở Run 3 xảy ra đồng loạt trên cả 7/7 kiến trúc độc lập, và kiểm định McNemar trên toàn bộ 3.000 mẫu kiểm thử xác nhận ý nghĩa thống kê (p=0.0161 và p<1e-6).",
        "evidence": "Bảng tiến trình 7/7 mô hình + Bảng Kiểm định McNemar",
    },
    {
        "id": 8, "group": "3. Thuật toán & Kết quả", "level": "Khó",
        "question": "Tại sao Weighted Soft (trọng số theo Val F1) lại thua Equal Soft (chia đều 1/7)? Và tại sao Top-3 Weighted lại không chọn DistilBERT-m?",
        "trap": "Lúng túng tưởng mình code sai trọng số.",
        "answer": "(1) Đây là minh chứng rõ nhất của Zero Data Leakage: nhóm chỉ nhìn val.csv (nơi PhoBERT 91.10%, Vi-ELECTRA 90.55%, mBERT 89.82% đứng đầu) để chọn Top-3 và trọng số, tuyệt đối không nhìn trộm test.csv. (2) Do val.csv cùng miền lệch lớp với train (IR=2.28), trọng số theo Val hơi thiên vị nhẹ các mô hình có độ lệch Val-Test lớn hơn (PhoBERT +2.97% vs DistilBERT-m +0.34%). Chia đều 1/7 đóng vai trò điều chuẩn chống quá khớp theo Val.",
        "evidence": "Bảng B7 (cột Val F1 vs Test Macro-F1 và ECE)",
    },
    {
        "id": 9, "group": "3. Thuật toán & Kết quả", "level": "Khó",
        "question": "Trong bài báo bạn viết Quality Gate đặt trọng số = 0 nếu Val F1 < 0.50, nhưng trong log Bước B4 mô hình ViDeBERTa-base bị collapse vẫn có trọng số 0.0253! Có mâu thuẫn không?",
        "trap": "Chối bỏ hoặc nói quên chưa kiểm tra bảng B4.",
        "answer": "(1) Trả lời thẳng đúng trình tự thực nghiệm: ở lần chạy B4 đầu tiên, hệ thống tính trọng số tỷ lệ Val F1 nên ViDeBERTa tự bị hạ xuống 2.53% (hầu như không ảnh hưởng tới 23.378 nhãn giả có Avg Conf 0.9682). (2) Chính từ bài học thực tế này, nhóm đã lập trình bổ sung cơ chế Quality Gate (min_model_val_f1=0.50) vào src/ensemble.py từ Vòng 2 trở đi và chuẩn hóa trong mô hình lý thuyết để hệ thống miễn nhiễm hoàn toàn khi tái chạy.",
        "evidence": "File src/ensemble.py + Bảng B4 (Avg Conf 0.9682, Avg Margin 0.9533)",
    },
    {
        "id": 10, "group": "3. Thuật toán & Kết quả", "level": "Khó",
        "question": "Mất cân bằng lớp IR=2.29 khá nhẹ, sao không dùng Class-Weighted Cross-Entropy hay Focal Loss cho nhanh mà phải chạy Pseudo-Labeling 3 vòng?",
        "trap": "Nói rằng Focal Loss không hiệu quả cho NLP.",
        "answer": "(1) Class-Weighted Loss hay Oversampling chỉ tái sử dụng lặp lại đúng 3.945 câu Environment cũ — không bổ sung thêm từ vựng hay cấu trúc câu mới nào, thậm chí dễ gây overfitting và tăng ECE. (2) Dữ liệu bài này còn gặp Covariate Length Shift (Test 44.3 từ vs Train 37.9 từ) và có sẵn 54.487 câu không nhãn dài hơn (48.3 từ). Selective Pseudo-Labeling giải quyết đồng thời 3 bài toán: thêm dữ liệu mới, thích nghi câu dài và cân bằng lớp.",
        "evidence": "Hình B2_02 (Length Shift) + Hình B6_02 (Cân bằng 3 tầng)",
    },
    {
        "id": 11, "group": "3. Thuật toán & Kết quả", "level": "CỰC GẮT",
        "question": "Theo chính bài báo Guo et al. (2017) bạn trích dẫn, chỉ cần dùng Temperature Scaling trên 1 mô hình DistilBERT-m cũng giảm ECE xuống ~0.02, cần gì chạy 7 mô hình Ensemble?",
        "trap": "Phản bác Guo et al. hoặc nói Temperature Scaling không giảm được ECE.",
        "answer": "(1) Hoàn toàn đồng ý về mặt toán học: Temperature Scaling giảm ECE rất tốt cho mô hình đơn lẻ (và nên áp dụng nếu triển khai bản 1 mô hình DistilBERT-m). (2) Điểm khác biệt then chốt: chia logit cho hằng số dương T>0 bảo toàn hoàn toàn argmax nên KHÔNG tăng được 0.01% Accuracy hay Macro-F1 nào (DistilBERT-m vẫn dừng ở 89.03%). Equal Soft (7M) mang lại hiệu ứng kép: vừa tăng Macro-F1 lên 90.09% (+1.06%, p=0.0161), vừa giảm ECE tự nhiên xuống 0.0226.",
        "evidence": "Hình B8_06 (Reliability Diagram) + Bảng McNemar (p=0.0161)",
    },
    {
        "id": 12, "group": "4. Mô hình hóa & Lỗi", "level": "Rất hay hỏi",
        "question": "Thời đại LLM (GPT-4o, Gemini, Qwen, Llama), tại sao bạn vẫn dùng các mô hình Encoder thế hệ BERT?",
        "trap": "Chê LLM kém chính xác hơn BERT.",
        "answer": "4 lý do triển khai thực tế: (1) Tốc độ & chi phí: DistilBERT-m chạy 3.085 câu/s (0.324 ms/câu), Ensemble 7M chạy 234 câu/s cục bộ với 0 đồng phí API — nhanh hơn gọi LLM hàng trăm lần. (2) Bảo mật on-premise: báo cáo tài chính/quản trị chưa công bố không thể gửi ra API bên ngoài. (3) Hiệu chuẩn xác suất rõ ràng (ECE=0.0226) phục vụ Human-in-the-Loop. (4) Bổ trợ chứ không loại trừ: hướng phát triển đề xuất dùng LLM gán nhãn ngữ cảnh đoạn dài rồi chưng cất về Encoder gọn nhẹ.",
        "evidence": "Hình B7_02 (Throughput) + Hình B8_05 (Pareto)",
    },
    {
        "id": 13, "group": "4. Mô hình hóa & Lỗi", "level": "Trung bình",
        "question": "Precision của lớp Governance chỉ đạt 82.54% (85 câu Irrelevant bị đoán nhầm sang Governance). Nếu triển khai thật khách hàng phàn nàn nhiều cảnh báo giả thì xử lý thế nào?",
        "trap": "Nói rằng 82.54% đã cao rồi nên không cần sửa.",
        "answer": "(1) Giải pháp ngắn hạn không cần huấn luyện lại: tối ưu ngưỡng quyết định riêng theo từng lớp (Class-wise Threshold Tuning) trên val.csv — đòi hỏi P(Governance) cao hơn mới gán nhãn, đổi một phần nhỏ Recall Governance đang rất cao (97.1%) lấy Precision cao hơn. (2) Giải pháp dài hạn: phân loại ở cấp đoạn văn + tiêu đề mục để phân biệt thuyết minh báo cáo tài chính (Irrelevant) với quản trị bền vững (Governance).",
        "evidence": "Hình B8_04 (Confusion Matrix) + Bảng B8_confusion_pairs.csv",
    },
    {
        "id": 14, "group": "5. Kỹ thuật & Triển khai", "level": "Trung bình",
        "question": "Đặt max_length = 256 token có cắt mất nội dung của các câu dài 454 từ (Test) hay 796 từ (Unlabeled) không?",
        "trap": "Nói rằng 256 token chứa hết 100% mọi câu.",
        "answer": "(1) Số liệu EDA B2 xác nhận 99.90% câu trên test (P99=133 từ ~190 token) và 99.60% trên unlabeled nằm trọn trong 256 token. (2) PhoBERT-base có giới hạn kiến trúc cứng là 256 token (max_position_embeddings=258); chọn 256 đảm bảo đầu vào công bằng 100% giữa cả 7 mô hình và tránh tăng chi phí Attention O(L^2) lên gấp 4 lần chỉ vì 0.1% câu ngoại lai.",
        "evidence": "Bảng thống kê độ dài Mục 3.3 + Hình B2_02",
    },
    {
        "id": 15, "group": "5. Kỹ thuật & Triển khai", "level": "Khó",
        "question": "Ensemble 7 mô hình nặng 988.4M tham số, tốn ~4GB VRAM và phức tạp bảo trì. Khi đi làm thực tế bạn khuyên doanh nghiệp triển khai bản nào?",
        "trap": "Cố khuyên doanh nghiệp nào cũng phải chạy đủ cả 7 mô hình.",
        "answer": "Tư vấn rõ theo Đường biên tối ưu Pareto (Hình B8_05): (1) API thời gian thực / hạ tầng nhỏ -> dùng DistilBERT-m Run 3 (89.03% Macro-F1, 0.324 ms/câu, nhanh hơn 13.2 lần, đã thừa hưởng sẵn tri thức của 2 vòng Ensemble Pseudo-Labeling). (2) Kiểm toán theo lô hàng quý cần độ tin cậy tối đa -> dùng Equal Soft 7M (90.09%, ECE=0.0226, xử lý 10.000 câu chỉ trong ~43 giây). (3) Bước sản xuất hóa tiếp theo -> Knowledge Distillation từ 7M về 1 bản DistilBERT-m duy nhất.",
        "evidence": "Hình B8_05 (Đường biên Pareto) + Bảng B7",
    },
]

groups_list = ["Tất cả 15 Gap"] + sorted(list({g["group"] for g in GAPS_BANK}))
state = {"pool": list(GAPS_BANK), "idx": 0, "show_ans": False, "mastered": set(), "review": set()}

dd_group = widgets.Dropdown(options=groups_list, description="Chủ đề:", layout=widgets.Layout(width="320px"))
btn_rand = widgets.Button(description="Rút câu ngẫu nhiên", button_style="primary", icon="random")
btn_next = widgets.Button(description="Câu tiếp theo", button_style="info", icon="arrow-right")
btn_toggle = widgets.Button(description="Mở / Ẩn gợi ý trả lời", button_style="warning", icon="eye")
btn_ok = widgets.Button(description="Đã nắm vững", button_style="success", icon="check")
btn_retry = widgets.Button(description="Cần luyện lại", button_style="danger", icon="repeat")
view = widgets.HTML(layout=widgets.Layout(width="98%", overflow="visible"))

def _render_card():
    if not state["pool"]:
        view.value = "<b>Không có câu hỏi trong nhóm đã chọn.</b>"
        return
    item = state["pool"][state["idx"] % len(state["pool"])]
    lvl_color = "#C62828" if "CỰC" in item["level"] else ("#E65100" if item["level"] == "Khó" else "#1565C0")
    progress_txt = f"Đã nắm vững: <b>{len(state['mastered'])}/15</b> &nbsp;|&nbsp; Cần luyện lại: <b>{len(state['review'])}</b>"
    status_tag = (
        "<span style='background:#E8F5E9;color:#1B5E20;padding:2px 8px;border-radius:4px;font-weight:bold;'>ĐÃ NẮM VỮNG</span>"
        if item["id"] in state["mastered"]
        else ("<span style='background:#FFEBEE;color:#C62828;padding:2px 8px;border-radius:4px;font-weight:bold;'>CẦN LUYỆN LẠI</span>" if item["id"] in state["review"] else "<span style='background:#ECEFF1;color:#455A64;padding:2px 8px;border-radius:4px;'>Chưa chấm</span>")
    )

    ans_html = ""
    if state["show_ans"]:
        ans_html = f"""
        <div style="margin-top:12px;background:#FFF8E1;border-left:5px solid #F57F17;padding:10px 14px;border-radius:5px;">
          <b style="color:#E65100;">Bẫy cần tránh:</b> {item['trap']}
        </div>
        <div style="margin-top:10px;background:#E8F5E9;border-left:5px solid #2E7D32;padding:12px 14px;border-radius:5px;">
          <b style="color:#1B5E20;">Gợi ý trả lời chuẩn học thuật:</b><br>{item['answer']}
        </div>
        <div style="margin-top:8px;font-size:13px;color:#37474F;">
          <b>Biểu đồ / Bằng chứng nên mở ra chứng minh:</b> <code>{item['evidence']}</code>
        </div>
        """
    else:
        ans_html = "<div style='margin-top:12px;padding:10px 14px;background:#F5F5F5;border-radius:5px;color:#666;font-style:italic;'>Hãy tự tập trả lời thành tiếng trong 30–60 giây, sau đó bấm nút <b>Mở / Ẩn gợi ý trả lời</b> để tự đối chiếu!</div>"

    view.value = f"""
    <div style="font-family:Segoe UI,Arial,sans-serif;border:1px solid #CFD8DC;border-radius:8px;padding:16px;background:#FFFFFF;line-height:1.55;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;font-size:13px;color:#455A64;">
        <div><b>Gap #{item['id']}</b> &nbsp;|&nbsp; {item['group']} &nbsp;|&nbsp; <span style='color:{lvl_color};font-weight:bold;'>Mức độ: {item['level']}</span> &nbsp;{status_tag}</div>
        <div>{progress_txt}</div>
      </div>
      <div style="font-size:16px;font-weight:bold;color:#1A237E;background:#E8EAF6;padding:12px 16px;border-radius:6px;border-left:5px solid #3949AB;">
        Giảng viên hỏi: "{item['question']}"
      </div>
      {ans_html}
    </div>
    """

def _on_group(change):
    if change["name"] == "value":
        val = change["new"]
        state["pool"] = list(GAPS_BANK) if val == "Tất cả 15 Gap" else [g for g in GAPS_BANK if g["group"] == val]
        state["idx"] = 0
        state["show_ans"] = False
        _render_card()

def _on_rand(_):
    state["idx"] = random.randrange(len(state["pool"]))
    state["show_ans"] = False
    _render_card()

def _on_next(_):
    state["idx"] = (state["idx"] + 1) % len(state["pool"])
    state["show_ans"] = False
    _render_card()

def _on_toggle(_):
    state["show_ans"] = not state["show_ans"]
    _render_card()

def _on_ok(_):
    gid = state["pool"][state["idx"] % len(state["pool"])]["id"]
    state["mastered"].add(gid)
    state["review"].discard(gid)
    _render_card()

def _on_retry(_):
    gid = state["pool"][state["idx"] % len(state["pool"])]["id"]
    state["review"].add(gid)
    state["mastered"].discard(gid)
    _render_card()

dd_group.observe(_on_group, names="value")
btn_rand.on_click(_on_rand)
btn_next.on_click(_on_next)
btn_toggle.on_click(_on_toggle)
btn_ok.on_click(_on_ok)
btn_retry.on_click(_on_retry)

display(widgets.VBox([
    widgets.HBox([dd_group, btn_rand, btn_next]),
    widgets.HBox([btn_toggle, btn_ok, btn_retry]),
    view,
], layout=widgets.Layout(width="100%", overflow="visible")))
_render_card()